# RF Board Setup Demo

This notebook demonstrates how to use the `rfboard` helper module to configure
on-board ADMV8818 bandpass filters and attenuators. These functions replace the
copy-pasted helper cells that were previously defined in each notebook.

# Initialize

In [ ]:
cfg_file = 'rfboard_demo.yml'  # Configuration file name
expt_path = '/tmp/rfboard_demo/'  # Experiment data path

## Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from qick import QickConfig

from slab_qick_calib.exp_handling.instrumentmanager import InstrumentManager
from slab_qick_calib.helpers import config, handy, rfboard

%load_ext autoreload
%autoreload 2

handy.config_figs()

## Set up new config

New configs automatically include RF board filter fields (`filter_fc`, `filter_bw`, `filter_type`)
in every channel group.

In [ ]:
new_config = False
new_folder = False

nqubits = 4
rfsoc_alias = 'bf1_soc'
t1_guess = 50
ip = '10.108.30.23'

configs_dir = os.path.join(os.getcwd(), '..', 'configs')
cfg_file_path = os.path.join(configs_dir, cfg_file)

if new_config or new_folder:
    if new_config:
        config.init_config(cfg_file_path, nqubits, type='full',
                           aliases=rfsoc_alias, t1=t1_guess, ip=ip)
    if not os.path.exists(expt_path):
        os.makedirs(expt_path)

print('Data will be stored in', expt_path)

### Migrate an existing config (optional)

If you have an older config that was created before the filter fields existed,
use `init_rfboard_section` to add them with bypass defaults. This is safe to
call on configs that already have the fields.

In [ ]:
# config.init_rfboard_section(cfg_file_path, nqubits)

## Connect to RFSoC

Before running, make sure a nameserver is running on the network.

In [ ]:
cfg_path = os.path.join(os.getcwd(), '..', 'configs', cfg_file)
auto_cfg = config.load(cfg_path)

im = InstrumentManager(ns_address=auto_cfg['aliases']['ip'])
print(im)

soc = im[auto_cfg['aliases']['soc']]
soccfg = QickConfig(soc.get_cfg())

cfg_dict = {'soc': soccfg, 'expt_path': expt_path, 'cfg_file': cfg_path, 'im': im}

# RF Board Configuration

All functions are imported from `slab_qick_calib.helpers.rfboard`.
Frequencies in the config are in MHz; the helpers handle MHz → GHz conversion internally.

## Bypass all filters

Start with a clean slate by disabling all on-board filters.

In [ ]:
gen_channels = [0, 1]  # DAC channels used
ro_channels = [0]      # ADC channels used

rfboard.bypass_all_filters(soc, gen_channels, ro_channels)

## Setup readout chain

Sets bandpass filters on ADC and DAC readout channels centered on the readout
frequency from config, and configures attenuators.

In [ ]:
qi = 0

ro_info = rfboard.setup_readout_chain(
    qi, soc, auto_cfg,
    bw_adc=500,     # ADC filter bandwidth (MHz)
    bw_dac=100,     # DAC filter bandwidth (MHz)
    atten1_dac=0,   # DAC attenuator 1 (dB)
    atten2_dac=0,   # DAC attenuator 2 (dB)
    atten_adc=30,   # ADC attenuator (dB)
)
print(ro_info)

## Setup qubit drive chain

Sets bandpass filter on the qubit DAC channel centered on `f_ge` from config.

In [ ]:
qi = 0

drive_info = rfboard.setup_qubit_drive_chain(
    qi, soc, auto_cfg,
    bw=1000,        # Filter bandwidth (MHz)
    atten1_dac=0,
    atten2_dac=0,
)
print(drive_info)

### Override center frequency

Use `center_freq` to set the filter around a frequency other than `f_ge`
(e.g., for spectroscopy searches).

In [ ]:
drive_info = rfboard.setup_qubit_drive_chain(
    qi, soc, auto_cfg,
    bw=2000,
    center_freq=6300,  # custom center in MHz
)
print(drive_info)

## Setup for multiple qubits

In [ ]:
qubit_list = [0, 1, 2]

for qi in qubit_list:
    rfboard.setup_readout_chain(qi, soc, auto_cfg, bw_adc=500, bw_dac=100, atten_adc=30)
    rfboard.setup_qubit_drive_chain(qi, soc, auto_cfg, bw=1000)
    print(f'Qubit {qi} RF chains configured')

## Low-level: `set_bandpass_rf`

For finer control — specify channels and frequencies directly.
Includes ADMV8818 frequency validation and a configurable settle time.

In [ ]:
f_lo, f_hi = rfboard.set_bandpass_rf(
    soc,
    gen_ch=0,       # DAC channel
    ro_ch=0,        # ADC channel
    fc_MHz=7000,    # center frequency
    bw_MHz=500,     # bandwidth
    tx_att1=0,      # optional TX attenuator 1
    tx_att2=0,      # optional TX attenuator 2
    ro_att=20,      # optional RX attenuator
    settle_ms=30,   # settle time after programming
)
print(f'Filter edges: {f_lo:.1f} - {f_hi:.1f} MHz')

# Persisting RF Settings to Config

Use `update_rfboard` to save filter settings so they can be restored later
with `apply_config_rf_settings`.

## Save filter settings

In [ ]:
qi = 0

# Save readout ADC filter settings
config.update_rfboard(cfg_path, 'adcs.readout', 'filter_fc', 7000, index=qi)
config.update_rfboard(cfg_path, 'adcs.readout', 'filter_bw', 500, index=qi)
config.update_rfboard(cfg_path, 'adcs.readout', 'filter_type', 'bandpass', index=qi)

# Save readout DAC filter settings
config.update_rfboard(cfg_path, 'dacs.readout', 'filter_fc', 7000, index=qi)
config.update_rfboard(cfg_path, 'dacs.readout', 'filter_bw', 100, index=qi)
config.update_rfboard(cfg_path, 'dacs.readout', 'filter_type', 'bandpass', index=qi)

# Save qubit DAC filter settings
auto_cfg = config.update_rfboard(cfg_path, 'dacs.qubit', 'filter_fc', 4500, index=qi)
config.update_rfboard(cfg_path, 'dacs.qubit', 'filter_bw', 1000, index=qi)
auto_cfg = config.update_rfboard(cfg_path, 'dacs.qubit', 'filter_type', 'bandpass', index=qi)

## Restore settings from config

Push all stored filter and attenuator settings to the hardware in one call.
Optionally pass `qubit=qi` to only configure one qubit's channels.

In [ ]:
auto_cfg = config.load(cfg_path)

# Apply all stored settings
rfboard.apply_config_rf_settings(soc, auto_cfg)

# Or just one qubit
# rfboard.apply_config_rf_settings(soc, auto_cfg, qubit=0)

## Verify config contents

In [ ]:
auto_cfg = config.load(cfg_path)

print('ADC readout filter_fc:  ', auto_cfg.hw.soc.adcs.readout.filter_fc)
print('ADC readout filter_bw:  ', auto_cfg.hw.soc.adcs.readout.filter_bw)
print('ADC readout filter_type:', auto_cfg.hw.soc.adcs.readout.filter_type)
print()
print('DAC readout filter_fc:  ', auto_cfg.hw.soc.dacs.readout.filter_fc)
print('DAC readout filter_bw:  ', auto_cfg.hw.soc.dacs.readout.filter_bw)
print('DAC readout filter_type:', auto_cfg.hw.soc.dacs.readout.filter_type)
print()
print('DAC qubit filter_fc:    ', auto_cfg.hw.soc.dacs.qubit.filter_fc)
print('DAC qubit filter_bw:    ', auto_cfg.hw.soc.dacs.qubit.filter_bw)
print('DAC qubit filter_type:  ', auto_cfg.hw.soc.dacs.qubit.filter_type)

# Active RF State Tracking

The `rfboard_active` section in the config tracks what each physical channel
is currently set to. This is useful when multiple qubits share the same
DAC/ADC channels and you need to know which qubit's settings are active.

## Migrate an existing config

Add the `rfboard_active` section to an older config that doesn't have it.
Safe to call multiple times — existing entries are preserved.

In [ ]:
# Add rfboard_active section to an existing config (idempotent)
# config.init_rfboard_active_section(cfg_path)

## Activate a qubit's RF settings

`activate_qubit_rf` reads the per-qubit ideal filter/attn settings from
the config arrays and programs the hardware. It updates `rfboard_active`
in memory (and on disk if `cfg_file` is given).

In [ ]:
qi = 0
auto_cfg = rfboard.activate_qubit_rf(qi, soc, auto_cfg, cfg_file=cfg_path)

## Inspect active RF state

Use `get_active_rf_state` to see what each physical channel is currently set to.

In [ ]:
# Full active state
print('Full active state:')
for ch_type, channels in rfboard.get_active_rf_state(auto_cfg).items():
    print(f'  {ch_type}:')
    for ch, state in channels.items():
        print(f'    ch {ch}: {dict(state)}')

# Single channel query
print('\nADC ch 0:', dict(rfboard.get_active_rf_state(auto_cfg, 'adc', 0)))

## Switch between qubits on shared channels

When multiple qubits share the same physical channel, `activate_qubit_rf`
reprograms the hardware and updates the active state to reflect which
qubit's settings are currently loaded.

In [ ]:
# Switch to qubit 1 — shared channels get reprogrammed
auto_cfg = rfboard.activate_qubit_rf(1, soc, auto_cfg, cfg_file=cfg_path)

# Verify the active state now shows qubit 1
print('After switching to qubit 1:')
for ch_type, channels in rfboard.get_active_rf_state(auto_cfg).items():
    print(f'  {ch_type}:')
    for ch, state in channels.items():
        print(f'    ch {ch}: qubit={state.get("qubit")}, fc={state.get("filter_fc")}')